# Show t_start × t_end grids for each source image

For each source image in `TARGET_FOLDER`, paste its `t_delta_<slug>/cells/*.png`
into one composite image and `imshow` it (single fast plot, nothing saved).

- **x-axis** = `t_start` (increasing left → right)
- **y-axis** = `t_end` (increasing bottom → top)

> **TEMP patch:** the display cell currently only plots the **first** source image.

> **Note:** Running this notebook embeds full-resolution PNGs in the `.ipynb` file (each plot can be ~10 MB). Clear outputs before saving or committing — **Cell → All Output → Clear**, or:
> `jupyter nbconvert --clear-output --inplace scripts/daniel_create_grid_image.ipynb`


In [ ]:
import os

# Use a writable matplotlib cache dir (silences the default-path warning).
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")

from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# t_start / t_end sweep values (matches scripts/run_grid_ablation.GRID_VALUES).
GRID_VALUES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

In [ ]:
def load_cell(cells_dir: Path, t_start: float, t_end: float):
    """Load a cell image from the given directory."""
    def _param_slug(param_name: str, value: float) -> str:
        return f"{param_name}_{value:.1f}".replace(".", "p")
    path = cells_dir / f"{_param_slug('t_start', t_start)}__{_param_slug('t_end', t_end)}.png"
    if not path.exists():
        return None
    with Image.open(path) as img:
        return img.convert("RGB")


def build_grid_image(cells_dir: Path, values, thumb_px: int = 96, gap: int = 3,
                     highlights=None, border_px: int = 5, t_delta: float = 0.0):
    """Paste all cells into one composite (t_start = x, t_end up the y-axis).

    ``highlights`` maps (t_start, t_end) -> border color; matching cells get a
    colored frame drawn on their edge.

    Cells with ``t_start - t_delta < 0`` are left white (not rendered).
    """
    highlights = highlights or {}
    t_delta = float(t_delta)
    n = len(values)
    stride = thumb_px + gap
    size = n * stride + gap  # gap also frames the outer edge
    canvas = Image.new("RGB", (size, size), "#ffffff")
    draw = ImageDraw.Draw(canvas)
    found = False
    for row, t_end in enumerate(reversed(values)):  # top row = largest t_end
        for col, t_start in enumerate(values):
            x = gap + col * stride
            y = gap + row * stride
            # if t_start - t_delta < 0:
            #     continue
            cell = load_cell(cells_dir, t_start, t_end)
            if cell is not None:
                found = True
                thumb = cell.resize((thumb_px, thumb_px), Image.Resampling.LANCZOS)
                canvas.paste(thumb, (x, y))
            color = highlights.get((round(t_start, 1), round(t_end, 1)))
            if color is not None:
                draw.rectangle(
                    [x, y, x + thumb_px - 1, y + thumb_px - 1],
                    outline=color,
                    width=border_px + 1,
                )
                draw.rectangle(
                    [x + border_px + 1, y + border_px + 1, x + thumb_px - border_px - 2, y + thumb_px - border_px - 2],
                    outline="white",
                    width=4,
                )
    return canvas if found else None

In [ ]:
# --- Parameters ----------------------------------------------------------- #
import json

TARGET_FOLDER = Path(
    "/data/home/mirick/ChordEdit/ablation_outputs/grid_metrics_sdturbo_top_20260626_104807"
)
THUMB_PX = 250  # per-cell size in the composite (source cells are 512x512)
GAP = 3         # light grey border (px) between cells
FIG_DPI = 200   # display resolution; figsize is derived from composite size

# (t_start, t_end) -> highlight border color.
HIGHLIGHTS = {
    (0.9, 0.3): "#1f77b4",
    (0.0, 0.0): "#000000",
    # (0.1, 0.0): "#ff7f0e",
}

# PIE-Bench mapping: sample_id -> {original_prompt, editing_prompt, ...}
MAPPING_PATH = Path("/data/home/mirick/datasets/PIE-Bench_v1/mapping_file.json")
MAPPING = json.loads(MAPPING_PATH.read_text())


def strip_brackets(text: str) -> str:
    return text.replace("[", "").replace("]", "").strip()

In [ ]:
values = list(GRID_VALUES)
n = len(values)
centers = [GAP + i * (THUMB_PX + GAP) + THUMB_PX / 2 for i in range(n)]

sample_dirs = sorted(p for p in TARGET_FOLDER.iterdir() if p.is_dir() and p.name != "plots")

for sample_dir in sample_dirs:
    for condition_dir in sorted(
        p for p in sample_dir.iterdir() if p.is_dir() and p.name.startswith("t_delta_")
    ):
        t_delta = float(condition_dir.name.replace("t_delta_", "").replace("p", "."))
        composite = build_grid_image(
            condition_dir / "cells",
            values,
            THUMB_PX,
            GAP,
            highlights=HIGHLIGHTS,
            t_delta=t_delta,
        )
        if composite is None:
            continue

        # Folder name: <prefix>_<name...>_<count>_<sample_id>.
        # e.g. "1_change_object_80_111000000000" -> id=111000000000, cat=1_change_object
        parts = sample_dir.name.split("_")
        sample_id = parts[-1]
        category = "_".join(parts[:-2])
        meta = MAPPING.get(sample_id, {})
        source_prompt = strip_brackets(meta.get("original_prompt", ""))
        target_prompt = strip_brackets(meta.get("editing_prompt", ""))
        title = f'Images for {sample_id} from {category}\nSource Prompt: "{source_prompt}"\nTarget Prompt: "{target_prompt}"'

        fig_w, fig_h = composite.width / FIG_DPI, composite.height / FIG_DPI
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=FIG_DPI)
        ax.imshow(composite, interpolation="none")
        ax.set_xticks(centers, [f"{v:.1f}" for v in values])
        ax.set_yticks(centers, [f"{v:.1f}" for v in reversed(values)])
        ax.set_xlabel(f"t_start, $\\delta={float(t_delta):.2f}$")
        ax.set_ylabel(f"t_end, $\\delta={float(t_delta):.2f}$")
        ax.set_title(title, fontsize=10)
